# 03 · LightGBM on the notebook-02 output

Trains LightGBM on `data/processed_hourly.parquet` and scores a holdout against the naive
baselines. Every setting lives in the first code cell, which is tagged `parameters` so
papermill can override it. The last cell runs the same notebook on the GX10 instead.
Solar geometry comes from `celine.forecasting.core.weather`, as in notebook 02.

In [ ]:
# What to train
TARGETS = ["grid_export", "grid_import"]   # grid_export only runs on devices with PV
SCOPE = "per_device"                       # "per_device" (model per device) | "pooled" (per target)
HORIZON = 48                               # forecast window in hours: 48 | 96 | 168
FEATURE_SET = "selected"                   # "selected" (selected_features.json) | "all" (full pool)
DEVICES = None                             # None: every device; or a list of ids

# Holdout: each device's last HOLDOUT_DAYS, N_ORIGINS daily origins forecasting HORIZON hours
HOLDOUT_DAYS = 14
N_ORIGINS = 7

# LightGBM
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30
VALID_FRACTION = 0.15                      # latest share of training rows, for early stopping
LEARNING_RATE = 0.05
NUM_LEAVES = 31
MAX_DEPTH = 7
MIN_DATA_IN_LEAF = 20
FEATURE_FRACTION = 0.8
BAGGING_FRACTION = 0.8
BAGGING_FREQ = 5
LAMBDA_L1 = 0.1
LAMBDA_L2 = 0.1
OBJECTIVE = {"grid_export": "regression", "grid_import": "tweedie"}
TWEEDIE_VARIANCE_POWER = 1.5
USE_MONOTONIC = True                       # physics priors on the weather features
NUM_THREADS = 0                            # 0: all cores
SEED = 42

# SARIMA baseline, next to the naive ones: fitted once per device and target on the last
# SARIMA_TRAIN_DAYS before the holdout, then re-filtered with the same parameters at each origin
SARIMA = True
SARIMA_ORDER = (1, 0, 1)                   # (p, d, q)
SARIMA_SEASONAL_ORDER = (1, 1, 1, 24)      # (P, D, Q, s): daily seasonality
SARIMA_TRAIN_DAYS = 28

# Output
RUN_NAME = None                            # None: "<SCOPE>-h<HORIZON>-<timestamp>"
LOG_MLFLOW = True                          # MLFLOW_TRACKING_URI if set, else runs/mlflow.db
MLFLOW_EXPERIMENT = "nb03-lightgbm"

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# numpy warns on all-NaN means, the intended value for a lag before the device existed.
warnings.filterwarnings("ignore", category=RuntimeWarning)
PARAMS = {k: v for k, v in dict(globals()).items() if k.isupper()}

NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())
DATA_DIR = NB_DIR / "data"
RUN_NAME = RUN_NAME or f"{SCOPE}-h{HORIZON}-{time.strftime('%Y%m%d-%H%M%S')}"
OUT_DIR = REPO_ROOT / "runs" / "nb03" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

PT = pd.read_parquet(DATA_DIR / "processed_hourly.parquet")
quality = pd.read_csv(DATA_DIR / "device_quality.csv").set_index("device_id")
HAS_PV = quality["has_pv"].astype(bool).to_dict()
SELECTED = json.loads((DATA_DIR / "selected_features.json").read_text())
ALL_DEVICES = sorted(DEVICES or PT["device_id"].unique())
PT = PT[PT["device_id"].isin(ALL_DEVICES)]
# One continuous hourly frame per device, so a lag is a plain index lookup.
DEV = {d: g.drop_duplicates("ts_hour", keep="last").set_index("ts_hour").sort_index().asfreq("h")
       for d, g in PT.groupby("device_id")}
print("run:", RUN_NAME, "| devices:", len(ALL_DEVICES), "| rows:", len(PT), "| output:", OUT_DIR)

In [ ]:
# Features that depend only on the target hour (as in notebook 02, cell 41).
from celine.forecasting.core.weather import _haurwitz_clearsky_ghi, solar_position

SITE_LAT, SITE_LON, LOCAL_TZ = 45.9167, 11.1667, "Europe/Rome"

HOLIDAYS = {pd.Timestamp(d).date() for d in [
    "2025-01-01", "2025-01-06", "2025-04-21", "2025-04-25", "2025-05-01", "2025-06-02",
    "2025-08-15", "2025-11-01", "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-06", "2026-04-25", "2026-05-01", "2026-06-02",
    "2026-08-15", "2026-11-01", "2026-12-08", "2026-12-25", "2026-12-26"]}


def off(ts):
    """Holiday or weekend."""
    return ts.date() in HOLIDAYS or ts.weekday() >= 5


# A bridge day is a working day next to a holiday whose other neighbour is also off.
BRIDGES = {(pd.Timestamp(h) + pd.Timedelta(days=s)).date() for h in HOLIDAYS for s in (-1, 1)
           if not off(pd.Timestamp(h) + pd.Timedelta(days=s))
           and off(pd.Timestamp(h) + pd.Timedelta(days=2 * s))}

HOURS = pd.date_range(PT["ts_hour"].min(), PT["ts_hour"].max(), freq="h")
WX = PT.groupby("ts_hour")[["global_tilted_irradiance", "cloud_cover", "temperature_2m"]].first()
WX = WX.reindex(HOURS)
local_date = pd.Series(HOURS.tz_convert(LOCAL_TZ).date, index=HOURS)
doy = HOURS.tz_convert(LOCAL_TZ).dayofyear.to_numpy()
_, cz = solar_position(HOURS, SITE_LAT, SITE_LON)
gti, temp = WX["global_tilted_irradiance"], WX["temperature_2m"]

EXTRA = pd.DataFrame(index=HOURS)
EXTRA["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
EXTRA["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
EXTRA["is_holiday"] = local_date.isin(HOLIDAYS).astype(int)
EXTRA["is_bridge"] = local_date.isin(BRIDGES).astype(int)
EXTRA["clearsky_ghi"] = _haurwitz_clearsky_ghi(cz)
EXTRA["gti_roll_3h"] = gti.rolling(3, min_periods=1).mean()
EXTRA["cloud_std_6h"] = WX["cloud_cover"].rolling(6, min_periods=2).std()
EXTRA["temp_lag_24h"] = temp.shift(24)
EXTRA["temp_roll_24h"] = temp.rolling(24, min_periods=6).mean()
EXTRA["gti_day_sum"] = gti.groupby(local_date).transform("sum")
by_day = temp.groupby(local_date)
EXTRA["temp_range_day"] = by_day.transform("max") - by_day.transform("min")

# Sunrise and sunset per local day, from a 15-minute elevation grid.
day = pd.Timedelta(days=1)
fine = pd.date_range(HOURS[0].floor("D") - day, HOURS[-1].ceil("D") + day, freq="15min")
fine_elevation, _ = solar_position(fine, SITE_LAT, SITE_LON)
lit = pd.DataFrame({"d": fine.tz_convert(LOCAL_TZ).date, "ts": fine})[fine_elevation > 0]
sun = lit.groupby("d")["ts"].agg(["min", "max"])
sunrise = pd.to_datetime(local_date.map(sun["min"]), utc=True)
sunset = pd.to_datetime(local_date.map(sun["max"]), utc=True)
hours = pd.Series(HOURS, index=HOURS)
EXTRA["day_length_h"] = (sunset - sunrise).dt.total_seconds() / 3600
EXTRA["hours_since_sunrise"] = (hours - sunrise).dt.total_seconds() / 3600
EXTRA["hours_to_sunset"] = (sunset - hours).dt.total_seconds() / 3600
print("EXTRA:", EXTRA.shape)

In [ ]:
BASE_CALENDAR = ["hour_sin", "hour_cos", "day_of_week", "month", "is_weekend"]
WEATHER_FROM_FRAME = ["global_tilted_irradiance", "shortwave_radiation", "cloud_cover",
                      "temperature_2m", "solar_elevation", "effective_solar_pv", "clearsky_index",
                      "heating_degree", "cooling_degree", "pv_temp_factor", "is_daylight",
                      "cloud_cover_diff", "ghi_ramp", "theoretical_prod"]
SAME_HOUR_DAYS = [1, 2, 3, 7, 14, 21, 28]
HISTORY = ([f"same_hour_{d}d" for d in SAME_HOUR_DAYS]
           + ["mean_same_hour_7d", "median_same_hour_7d", "diff_1d", "diff_7d", "roll_24h_mean",
              "roll_24h_std", "roll_24h_max", "value_at_origin", "prev_day_total", "zero_share_7d"])


def feature_list(target):
    """The model's input columns for a target, per FEATURE_SET and SCOPE."""
    if FEATURE_SET == "selected":
        names = list(SELECTED[target])
    else:
        names = BASE_CALENDAR + list(EXTRA.columns) + WEATHER_FROM_FRAME
        names += [f"{target}_{n}" for n in HISTORY]
    return names + ["horizon"] + (["device_id"] if SCOPE == "pooled" else [])


def build(device, target, t, h):
    """Feature rows for target hours `t`, each forecast `h` hours ahead.

    Calendar and weather are read at t (perfect-forecast assumption, as in 02); every
    history feature is read at or before the origin t - h, so nothing leaks at any horizon.
    A same-hour lag of d days is only observable while 24 d >= h; shorter ones fall back
    to the most recent observable day (ceil(h / 24) days back).
    """
    dev, h = DEV[device], np.asarray(h)
    s = dev[target]

    def back(series, hours_back):
        return series.reindex(t - pd.to_timedelta(hours_back, unit="h")).to_numpy()

    X = dev.reindex(t)[BASE_CALENDAR + WEATHER_FROM_FRAME].reset_index(drop=True)
    X[list(EXTRA.columns)] = EXTRA.reindex(t).to_numpy()
    X["horizon"] = h
    X["device_id"] = pd.Categorical([device] * len(t), categories=ALL_DEVICES)
    p = f"{target}_"
    for d in SAME_HOUR_DAYS:
        X[p + f"same_hour_{d}d"] = back(s, 24 * np.maximum(d, np.ceil(h / 24)))
    week = np.column_stack([np.where(24 * d >= h, back(s, 24 * d), np.nan) for d in range(1, 8)])
    X[p + "mean_same_hour_7d"] = np.nanmean(week, axis=1)
    X[p + "median_same_hour_7d"] = np.nanmedian(week, axis=1)
    X[p + "diff_1d"] = X[p + "same_hour_1d"] - X[p + "same_hour_2d"]
    X[p + "diff_7d"] = X[p + "same_hour_7d"] - X[p + "same_hour_14d"]
    roll = s.rolling(24, min_periods=12)
    zero_share = (s == 0).where(s.notna()).astype(float).rolling(168, min_periods=24).mean()
    at_origin = {"roll_24h_mean": roll.mean(), "roll_24h_std": roll.std(),
                 "roll_24h_max": roll.max(), "value_at_origin": s, "prev_day_total": roll.sum(),
                 "zero_share_7d": zero_share}
    for name, series in at_origin.items():
        X[p + name] = back(series, h)
    return X, s.reindex(t).to_numpy()

In [ ]:
# Sparse training horizons; `horizon` is a feature, so every hour 1..HORIZON is scored.
TRAIN_H = [h for h in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 21, 24, 30, 36, 42, 48,
                       60, 72, 84, 96, 120, 144, 168] if h <= HORIZON]
PAIRS = [(d, t) for t in TARGETS for d in ALL_DEVICES if t != "grid_export" or HAS_PV.get(d, False)]
TRAIN_END = {d: DEV[d].index[-1] - pd.Timedelta(days=HOLDOUT_DAYS) for d in ALL_DEVICES}
MONOTONIC = {  # (positive, negative) physics priors, as in notebook 02
    "grid_export": (["global_tilted_irradiance", "shortwave_radiation", "effective_solar_pv",
                     "clearsky_index", "solar_elevation"], ["cloud_cover"]),
    "grid_import_pv": (["heating_degree", "cloud_cover"],
                       ["effective_solar_pv", "clearsky_index", "solar_elevation"]),
    "grid_import_no_pv": (["heating_degree"], []),
}


def fit(devices, target):
    """One booster on the training rows of `devices`, early-stopped on the latest rows."""
    parts = []
    for d in devices:
        dev = DEV[d]
        t = dev.index[(dev.index <= TRAIN_END[d]) & dev[target].notna()].repeat(len(TRAIN_H))
        X, y = build(d, target, t, np.tile(TRAIN_H, len(t) // len(TRAIN_H)))
        parts.append((X[feature_list(target)], y, t))
    X = pd.concat([p[0] for p in parts], ignore_index=True)
    y = np.concatenate([p[1] for p in parts])
    order = np.argsort(np.concatenate([p[2].asi8 for p in parts]), kind="stable")
    X, y = X.iloc[order], y[order]
    params = dict(objective=OBJECTIVE[target], learning_rate=LEARNING_RATE,
                  num_leaves=NUM_LEAVES, max_depth=MAX_DEPTH, min_data_in_leaf=MIN_DATA_IN_LEAF,
                  feature_fraction=FEATURE_FRACTION, bagging_fraction=BAGGING_FRACTION,
                  bagging_freq=BAGGING_FREQ, lambda_l1=LAMBDA_L1, lambda_l2=LAMBDA_L2,
                  num_threads=NUM_THREADS, seed=SEED, verbose=-1)
    if params["objective"] == "tweedie":
        params["tweedie_variance_power"] = TWEEDIE_VARIANCE_POWER
        if y.sum() <= 0:  # tweedie needs a positive label sum
            params["objective"] = "regression"
    if USE_MONOTONIC:
        # Pooled import mixes PV and non-PV devices, so it keeps the heating prior only.
        pv = len(devices) == 1 and HAS_PV.get(devices[0], False)
        key = target if target == "grid_export" else f"grid_import_{'pv' if pv else 'no_pv'}"
        pos, neg = MONOTONIC[key]
        params["monotone_constraints"] = [1 if c in pos else -1 if c in neg else 0
                                          for c in X.columns]
    cut = int(len(X) * (1 - VALID_FRACTION))
    train = lgb.Dataset(X.iloc[:cut], y[:cut])
    valid = lgb.Dataset(X.iloc[cut:], y[cut:], reference=train)
    return lgb.train(params, train, NUM_BOOST_ROUND, valid_sets=[valid],
                     callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])


t0 = time.time()
if SCOPE == "per_device":
    GROUPS = {(d, t): [d] for d, t in PAIRS}
else:
    GROUPS = {("pooled", t): [d for d, tt in PAIRS if tt == t] for t in TARGETS}
MODELS = {}
for (key, target), devices in GROUPS.items():
    MODELS[key, target] = fit(devices, target)
    print(f"{key:>16} {target}: {MODELS[key, target].num_trees()} trees  [{time.time() - t0:.0f}s]")
TRAIN_SECONDS = time.time() - t0

In [ ]:
# Daily origins after each device's training data, while a full HORIZON of actuals follows.
ORIGINS = {d: [o for k in range(N_ORIGINS)
               if (o := TRAIN_END[d].ceil("D") + pd.Timedelta(days=k))
               + pd.Timedelta(hours=HORIZON) <= DEV[d].index[-1]] for d in ALL_DEVICES}
BASELINES = ["naive168", "naive24"] + ["sarima"] * SARIMA


def sarima(d, target):
    """SARIMA forecasts {origin: HORIZON values} and an error message (None when it fits)."""
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from threadpoolctl import threadpool_limits

    warnings.filterwarnings("ignore")  # convergence chatter
    s, window = DEV[d][target], pd.Timedelta(days=SARIMA_TRAIN_DAYS)
    # One BLAS thread per worker: the matrices are tiny, so more threads only fight each other.
    with threadpool_limits(1):
        try:
            fit = SARIMAX(s[TRAIN_END[d] - window:TRAIN_END[d]], order=tuple(SARIMA_ORDER),
                          seasonal_order=tuple(SARIMA_SEASONAL_ORDER)).fit(disp=False)
            return {o: fit.apply(s[o - window:o]).forecast(HORIZON).to_numpy()
                    for o in ORIGINS[d]}, None
        except Exception as e:  # e.g. a flat series
            return {}, f"{type(e).__name__}: {e}"


SARIMA_FC = {}
if SARIMA:
    import multiprocessing
    from concurrent.futures import ProcessPoolExecutor

    t0 = time.time()
    # Forked workers see DEV and TRAIN_END as they are; each (device, target) is one fit.
    with ProcessPoolExecutor(NUM_THREADS or None,
                             mp_context=multiprocessing.get_context("fork")) as pool:
        for pair, (fc, err) in zip(PAIRS, pool.map(sarima, *zip(*PAIRS))):
            SARIMA_FC[pair] = fc
            if err:
                print("SARIMA failed, no baseline for", *pair, "|", err)
    print(f"SARIMA: {len(PAIRS)} fits in {time.time() - t0:.0f}s")

rows = []
for d, target in PAIRS:
    model = MODELS[d if SCOPE == "per_device" else "pooled", target]
    s = DEV[d][target]
    for origin in ORIGINS[d]:
        h = np.arange(1, HORIZON + 1)
        t = origin + pd.to_timedelta(h, unit="h")
        X, actual = build(d, target, t, h)
        rows.append(pd.DataFrame({
            "device_id": d, "target": target, "origin": origin, "horizon": h, "ts_hour": t,
            "actual": actual,
            "model": np.maximum(0, model.predict(X[feature_list(target)])),
            "naive168": s.reindex(t - pd.Timedelta(hours=168)).to_numpy(),
            # yesterday's same hour, or the most recent one observable at the origin
            "naive24": s.reindex(t - pd.to_timedelta(24 * np.ceil(h / 24), unit="h")).to_numpy(),
            **({"sarima": np.maximum(0, SARIMA_FC[d, target].get(origin, np.full(HORIZON, np.nan)))}
               if SARIMA else {})}))
RES = pd.concat(rows, ignore_index=True).dropna(subset=["actual"])
RES["band"] = pd.cut(RES["horizon"], [0, 8, 24, 48, 96, 168],
                     labels=["1-8", "9-24", "25-48", "49-96", "97-168"])


def mae(g):
    """MAE of the model and the baselines; skill vs a baseline only where its MAE > 0.01."""
    m = {c: (g[c] - g["actual"]).abs().mean() for c in ["model", *BASELINES]}
    for name, c in [("skill168", "naive168"), ("skill_sarima", "sarima")][:1 + SARIMA]:
        m[name] = 1 - m["model"] / m[c] if m[c] > 0.01 else np.nan
    return pd.Series(m)


overall = RES.groupby("target").apply(mae).assign(band="all").set_index("band", append=True)
SUMMARY = pd.concat([RES.groupby(["target", "band"], observed=True).apply(mae), overall])
SUMMARY = SUMMARY.sort_index()
PER_DEVICE = RES.groupby(["target", "device_id"]).apply(mae)
RES.to_csv(OUT_DIR / "holdout.csv", index=False)
SUMMARY.to_csv(OUT_DIR / "summary.csv")
PER_DEVICE.to_csv(OUT_DIR / "per_device.csv")
print(f"trained in {TRAIN_SECONDS:.0f}s | holdout rows {len(RES):,} | MAE in kWh/h")
display(SUMMARY.round(4))
display(PER_DEVICE.round(4))

In [ ]:
fig, axes = plt.subplots(len(TARGETS), 2, figsize=(15, 4.5 * len(TARGETS)),
                         squeeze=False)
for row, target in zip(axes, TARGETS, strict=True):
    gain = sum(pd.Series(m.feature_importance("gain"), index=m.feature_name())
               for (_, t), m in MODELS.items() if t == target)
    gain.nlargest(20).sort_values().plot.barh(ax=row[0], color="#3b6fb6")
    row[0].set(title=f"{target}: top-20 features by total gain", xlabel="gain")
    one = RES[RES["target"] == target]
    one = one[one["device_id"] == one["device_id"].iloc[0]]
    one = one[one["origin"] == one["origin"].min()]
    row[1].plot(one["ts_hour"], one["actual"], color="#222222", lw=2, label="actual")
    row[1].plot(one["ts_hour"], one["model"], color="#3b6fb6", lw=2, label="LightGBM")
    row[1].plot(one["ts_hour"], one["naive168"], color="#999999", lw=1.5, ls="--",
                label="naive 168 h")
    if SARIMA:
        row[1].plot(one["ts_hour"], one["sarima"], color="#d9822b", lw=1.5, label="SARIMA")
    start = one["origin"].iloc[0]
    row[1].set(title=f"{target}: {one['device_id'].iloc[0]} from {start:%Y-%m-%d %H:%M} UTC",
               ylabel="kWh/h")
    row[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "report.png", dpi=110)

In [ ]:
if LOG_MLFLOW:
    import mlflow

    default_uri = f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}"
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", default_uri))
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params({**PARAMS, "host": os.uname().nodename,
                           "train_seconds": round(TRAIN_SECONDS)})
        mlflow.log_metrics({f"{metric}.{t}.{band}": v for (t, band), r in SUMMARY.iterrows()
                            for metric, v in r.items() if pd.notna(v)})
    print("logged to", mlflow.get_tracking_uri())

In [ ]:
os.environ.get("MLFLOW_TRACKING_URI")

In [ ]:
# Run this notebook on the GX10 instead of here: set the parameters at the top, save the
# notebook, run the setup cell (the second one), then uncomment one call at the bottom.
# The job runs in a Docker container with a memory cap, because the box freezes if it runs
# out of memory (the Qwen server already holds about 65 GB). It keeps going if the laptop
# disconnects; results and the MLflow db land in runs/ on the box.
import shlex
import subprocess

GX10_HOST = os.environ.get("GX10_HOST", "sa@192.168.1.128")
GX10_DIR = "celine-forecasting"                     # on the box, relative to the home folder
GX10_MEMORY, GX10_CPUS = "24g", 12
GX10_IMAGE = "ghcr.io/astral-sh/uv:python3.12-bookworm"  # full image: LightGBM needs libgomp


def gx10_run():
    """Copy the repo and 02's outputs to the box, then start this notebook there."""
    job = time.strftime("%Y%m%d-%H%M%S")
    files = subprocess.run(["git", "ls-files", "-co", "--exclude-standard"], cwd=REPO_ROOT,
                           check=True, capture_output=True, text=True).stdout
    subprocess.run(["rsync", "-az", "--files-from=-", "./", f"{GX10_HOST}:{GX10_DIR}/"],
                   input=files, cwd=REPO_ROOT, check=True, text=True)
    inputs = [str(DATA_DIR / f) for f in
              ["processed_hourly.parquet", "device_quality.csv", "selected_features.json"]]
    subprocess.run(["rsync", "-az", *inputs, f"{GX10_HOST}:{GX10_DIR}/notebooks/data/"], check=True)
    # Every parameter from the top cell travels with the job, with the box's core count.
    params = json.dumps({**PARAMS, "NUM_THREADS": GX10_CPUS})
    remote = (
        f"cd {GX10_DIR} && mkdir -p runs/jobs/{job} && "
        f"nohup docker run --rm --name nb03-{job} --memory {GX10_MEMORY} --cpus {GX10_CPUS} "
        "--user $(id -u):$(id -g) -v $PWD:/w -w /w -e HOME=/tmp "
        "-e UV_CACHE_DIR=/w/.uv-cache -e UV_PROJECT_ENVIRONMENT=/w/.venv-docker "
        f"{GX10_IMAGE} uv run --frozen --extra notebooks --extra mlflow papermill "
        f"notebooks/03_lightgbm.ipynb runs/jobs/{job}/03_lightgbm.ipynb --cwd notebooks "
        f"-y {shlex.quote(params)} > runs/jobs/{job}/log.txt 2>&1 < /dev/null &")
    subprocess.run(["ssh", GX10_HOST, remote], check=True)
    print(f"started nb03-{job} on {GX10_HOST}")


def gx10_status():
    """Jobs still running, and the last lines of the latest log."""
    remote = ("docker ps --filter name=nb03- --format '{{.Names}}  {{.Status}}'; "
              f"cd {GX10_DIR} && log=$(ls -td runs/jobs/*/ | head -1)log.txt && echo $log && "
              "tail -c 600 $log | tr '\r' '\n' | tail -n 4")
    print(subprocess.run(["ssh", GX10_HOST, remote], capture_output=True, text=True).stdout)


def gx10_pull():
    """Copy runs/ from the box to runs/gx10/ here, then its MLflow runs to our tracking server."""
    local = REPO_ROOT / "runs" / "gx10"
    subprocess.run(["rsync", "-az", f"{GX10_HOST}:{GX10_DIR}/runs/", str(local)], check=True)
    gx10_to_mlflow(local / "mlflow.db")


def gx10_to_mlflow(db):
    """Copy the box's runs (params, metrics, tags) to MLflow here, skipping those already copied."""
    from mlflow import MlflowClient
    from mlflow.entities import Metric, Param

    src = MlflowClient(f"sqlite:///{db}")
    uri = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}")
    dst = MlflowClient(uri)
    copied = 0
    for exp in src.search_experiments():
        target = dst.get_experiment_by_name(exp.name)
        exp_id = target.experiment_id if target else dst.create_experiment(exp.name)
        # The box's run id travels as a tag, so pulling twice never copies a run twice.
        done = {r.data.tags.get("gx10.run_id") for r in dst.search_runs([exp_id], max_results=50000)}
        for run in src.search_runs([exp.experiment_id], max_results=50000):
            if run.info.run_id in done:
                continue
            tags = {**run.data.tags, "gx10.run_id": run.info.run_id, "gx10.host": GX10_HOST}
            new = dst.create_run(exp_id, start_time=run.info.start_time, tags=tags,
                                 run_name=run.info.run_name).info.run_id
            metrics = [Metric(m.key, m.value, m.timestamp, m.step) for k in run.data.metrics
                       for m in src.get_metric_history(run.info.run_id, k)]
            params = [Param(k, v) for k, v in run.data.params.items()]
            for i in range(0, len(metrics), 1000):  # log_batch caps: 1000 metrics, 100 params
                dst.log_batch(new, metrics=metrics[i:i + 1000])
            for i in range(0, len(params), 100):
                dst.log_batch(new, params=params[i:i + 100])
            dst.set_terminated(new, run.info.status, run.info.end_time)
            copied += 1
    print(f"copied {copied} new run(s) from {db} to {uri}")


In [ ]:
# gx10_run()
# gx10_status()
gx10_pull()